# Vaja 11: Numerično integriranje

In [ ]:
import numpy as np
import sympy as sym
import matplotlib.pyplot as plt
from scipy.integrate import newton_cotes

**Vprašanje 1:** Z uporabo orodij paketa ``sympy`` simbolno določite vrednost statičnega momenta prereza v obliki četrtine kroga na sliki:

$$S_{xx} = \int_{A}y~\text{d}A=\int_0^r \int_0^{\pi/2}y(\varphi)~r~\text{d}\varphi~\text{d}r = \int_{0}^{\pi/2}\frac{r^3}{3}\, \sin (\varphi)\, \text{d}\varphi$$

<img src="def.png" style="width:340px">

Pripravite tudi numerično funkcijo odvisnosti integranda $f(\varphi) = \Big(\frac{r^3}{3}\, \sin (\varphi) \Big)$ pri podani vrednosti polmera $R$.

*Podatki:*

$R = 7.5$ mm

In [ ]:
# Sympy, vprašanje 1
import sympy as sym
r, p = sym.symbols('r varphi')
f_sym = r**3/3 * sym.sin(p)
S_xx_sym = sym.integrate(f_sym, (p, 0, sym.pi/2)).subs(r, 7.5)
S_xx_sym

**Vprašanje 2:** Uporabite numerično funkcijo, pripravljeno pri prejšnji nalogi, in izračunajte ter grafično prikažite vrednosti integranda iz prve naloge $\Big(f(\varphi) = \frac{r^3}{3}\sin (\varphi) \Big)$ pri 100 diskretnih točkah na intervalu $\varphi \in [0, \pi/2]$. Izračunajte vrednost določenega integrala $S$ še numerično, s pomočjo trapezne in Simpsonove 1/3 metode iz paketov ``numpy`` oziroma ``scipy``.

$$ S = \int_{0}^{\pi/2}\frac{r^3}{3}\sin (\varphi)\text{d}\varphi$$

---

#### Trapezno pravilo

#### Sestavljeno trapezno pravilo

**Vprašanje 3:** Z uporabo lastne implementacije trapeznega pravila (osnovnega, ne sestavljenega) izračunajte določeni integral

$$I = \int_{0}^{2} -0.5x^3 - x^2 + 8 ~\text{d}x$$

Dobljeno vrednost primerjajte z rezultatom funkcije ``scipy.integrate.trapezoid``, kjer opazovan interval razdelite na 10 ekvidistantnih odsekov.

---

#### Izboljšan približek z Richardsonovo ekstrapolacijo

**Vprašanje 4:** Uporabite Richardsonovo ekstrapolacijo in določite natančnejšo vrednost integrala iz prejšnje naloge tako, da opazovan interval $x \in [0, 2]$ najprej razdelite na 2, nato še na 4 ekvidistantne segmente in uporabite sestavljeno trapezno metodo iz paketa ``numpy``.

**Vprašanje 5:** Izrišite potek absolutne vrednosti napake numerične integracije s sestavljenim trapeznim pravilom za primer iz prejšnje naloge, ko število diskretnih točk na $x$ osi povečujete od 2 do 10 točk. Napako integracije določite tako, da od numerično dobljene vrednosti odštejete pravo vrednost integrala, izračunano simbolno. 

In [ ]:
# Podatki, vprašanje 5
x_ = sym.symbols('x')
y = -0.5*x_**3 -x_**2 + 8
I_prav = sym.integrate(y, (x_, 0, 2)).evalf(15)

---

#### Simpsonovo 1/3 pravilo

#### Sestavljeno Simpsonovo 1/3 pravilo

**Vprašanje 6:** Določen integral iz prejšnje naloge izračunajte z lastno kodo Simpsonove 1/3 metode (osnovne, ne sestavljene).

$$I = \int_{0}^{2} -0.5x^3 - x^2 + 8 ~\text{d}x$$

Dobljeno vrednost primerjajte z rezultatom funkcije ``scipy.integrate.simpson``, kjer opazovan interval razdelite na 7 ekvidistantnih točk.

In [ ]:
# Prikaz Simpsonove 1/3 metode
from scipy.interpolate import lagrange
x_3 = np.linspace(0, 2, 3)
x_vec = np.linspace(0, 2, 100)
interpolacija = lagrange(x_3, fun(x_3))
plt.fill_between(x_vec, interpolacija(x_vec), label='interpolacija 2. reda  (Simpson 1/3)')
plt.plot(x_vec, fun(x_vec), 'k', label='funkcija')
plt.plot(x_3, fun(x_3), 'ro', label='točke, Simpson')
plt.legend()

#### Simpsonovo 3/8 pravilo

##### Uporaba ``scipy.integrate.newton_cotes``:

In [ ]:
# Uporaba Newton-Cotes uteži:
fun = lambda x: -0.5*x**3 - x**2 + 8
a, b = 0, 2
w = newton_cotes(3)[0]
x, h = np.linspace(a, b, len(w), retstep=True)
y = fun(x)
I = h*w @ y

---

**Vprašanje 7:** Podane so izmerjene vrednosti pospeška avtomobila v smeri vožnje v času. Z uporabo kumulativne trapezne metode (``scipy.integrate.cumulative_trapezoid``) določite vrednosti hitrosti in opravljene poti avtomobila v opazovanem časovnem intervalu. 

Avto pri času $t = 0$ miruje (namig: argument ``initial``).

In [ ]:
# Podatki, vprašanje 7
t = np.linspace(0, 7, 10) # s
a = np.array([ 3.47,  4.03,  4.36,  4.48,  4.27,  3.78,  3.19,  2.73,  2.54,  2.53]) # m/s^2

---

### Gaussova integracijska metoda (Gaussova kvadratura)

##### Uporaba ``scipy.special.legendre``

In [ ]:
# Uporaba `scipy.special.legendre` pri Gaussovi integraciji:
from scipy.special import legendre
vozlišč = 1
xi = legendre(vozlišč).weights[:,0]
Ai = legendre(vozlišč).weights[:,1]

In [ ]:
# Transformacija integracijskih mej pri Gaussovi integraciji:
def meje_gauss(xi, a, b):
    """
    Transformacija integracijskih mej Gaussove integracije iz `xi` na interval [a, b].
    """
    return (b+a)/2 + (b-a)/2*xi

**Vprašanje 8:** Z uporabo Gaussove integracije z dvema vozliščema določite vrednost integrala:

$$I = \int_{0}^{2} -0.5x^3 - x^2 + 8 ~\text{d}x$$

Vrednosti vozlišč in pripadajočih uteži določite z uporabo ``scipy.special.legendre``. Ne pozabite na transformacijo integracijskih mej!

$$x=\frac{b+a}{2}+\frac{b-a}{2}\xi$$

$$I \approx \frac{b-a}{2}\sum_i A_i\,f(x_i)$$

**Vprašanje 9:** Z uporabo sestavljene trapezne metode iz `scipy` in metode Gaussove integracije (``scipy.integrate.quad``) izračunajte vrednosti integralov:

$$I_1 = \int_{0}^{1}\frac{\sin(x)}{x} \text{d}x$$

$$I_2 = \int_{1}^{2}\frac{\sin(x)}{x} \text{d}x$$

$$I_3 = \int_{-0.5}^{0.5}\frac{\sin(x)}{x} \text{d}x$$

Pri uporabi trapezne metode razdelite opazovan interval na 10 točk.

**Vprašanje 10:** Integral:

$$I = \int_{-0.5}^{0.5}\frac{\sin(x)}{x} \text{d}x$$

izračunajte še z uporabo funkcije ``scipy.integrate.fixed_quad``, ki ji lahko podamo tudi parameter števila vozlišč pri integraciji, ``n``. Najprej uporabite integracijo z enim vozliščem, nato z dvema in nazadnje še s tremi vozlišči.